In [ ]:
from dask.distributed import LocalCluster, Client
cluster = LocalCluster(n_workers=4,
                       threads_per_worker=4,
                       memory_target_fraction=0.95,
                       memory_limit='8GB')
client = Client(cluster)
client

In [ ]:
import dask.dataframe as dd
import pandas as pd
import re

In [ ]:
df = dd.read_csv("archive-cc-2017-nc.csv")
df

In [ ]:
%%time
df.head()

In [ ]:
%%time
df[df.archive_id.isin(['FOXNEWSW_20170224_160000_Happening_Now', 'MSNBCW_20120726_050000_The_Last_Word'])].compute()

In [ ]:
# Define a function for string replacement
def replace_strings(column):
    try:
        return column.apply(lambda x: x if pd.isna(x) else re.sub("b'(.*)'", '\\1', x))
    except Exception as e:
        return column

In [ ]:
%%time
first = True
OUTPUT_FILE = '../shared/archive-cc-2017-all-nc.csv.gz'
for i, adf in enumerate(pd.read_csv('../shared/archive-cc-2017.csv.gz', chunksize=1000)):
    print(i)
    # FIXME: don't need for 2017
    #adf = adf.apply(lambda col: replace_strings(col), axis=0)
    idents = adf.identifier.tolist()
    cdf = df[df.archive_id.isin(idents)].compute()
    #print(len(cdf))
    adf = pd.merge(adf, cdf, how='left', left_on='identifier', right_on='archive_id')
    del adf['archive_id']
    if first:
        adf.to_csv(OUTPUT_FILE, index=False, header=first, compression='gzip')
        first = False
    else:
        adf.to_csv(OUTPUT_FILE, mode='a', index=False, header=first, compression='gzip')
    #if i >= 1:
    #    break